# Análisis de Fuentes de Datos — Débitos Recurrentes

## Prueba Técnica Bancolombia — Cartera en Mora Temprana

---

**Autor:** Jhon Fredy Correa Gomez  
**Email:** jonfredi12@gmail.com  
**Fecha:** Abril 2026

---

### Propósito

Este notebook documenta el análisis exploratorio de las **6 fuentes de datos** entregadas para la prueba técnica, el proceso de unificación en un modelo analítico de granularidad **cliente × obligación × periodo**, y la caracterización del dataset consolidado resultante.

### Objetivo de Negocio

Identificar qué obligaciones en mora temprana (1–30 días) tienen alta probabilidad de pagarse **exclusivamente mediante débito automático con recurrencia ≥ 40%**, con el fin de:

- Reducir costos de gestión de cobranza sobre esas obligaciones
- Priorizar recursos de cobranza hacia obligaciones con menor probabilidad de auto-pago
- Mejorar la eficiencia operativa del área de recaudo

### Variable Respuesta

| Clase | Definición |
|---|---|
| `var_rta = 1` | La obligación se paga **únicamente** por débito **Y** la recurrencia de ese canal es **≥ 40%** |
| `var_rta = 0` | Existen pagos por otros canales o combinación de canales (aunque haya débito, si no es exclusivo → clase 0) |

> **Implicación crítica:** La clase 1 exige DOS condiciones simultáneas: exclusividad de canal + frecuencia mínima.

### Estructura del Notebook

1. Configuración e importaciones
2. Supuestos de negocio declarados
3. Análisis individual de cada fuente CSV
4. Proceso de unificación paso a paso
   - 4.1 Tratamiento de duplicados en las fuentes CSV
   - 4.2 Cobertura de obligaciones entre fuentes
   - 4.3 Ejecución del modelo analítico unificado
   - 4.4 Validación de integridad del join
5. Caracterización del dataset consolidado
6. Split temporal Train / Test / OOT
7. Conclusiones y alertas

## 1. Configuración e Importaciones

In [ ]:
! pip install ipykernel

: 

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

# Módulo de preparación de datos — modelo analítico de débitos recurrentes
from src.dataset.data_preparation import (
    JOIN_KEYS,
    OOT_START,
    TARGET_COL,
    TEST_END,
    TEST_START,
    TRAIN_END,
    build_analytical_model,
    get_feature_columns,
    load_raw_sources,
    split_train_test_oot,
    validate_join_integrity,
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

DATALAKE = Path('../datalake')
FIGURES  = Path('../figures')
FIGURES.mkdir(exist_ok=True)

print('Configuración completada.')
print(f'Datalake : {DATALAKE.resolve()}')
print(f'Figuras  : {FIGURES.resolve()}')

: 

## 2. Supuestos de Negocio Declarados

Toda decisión de ingeniería de datos debe estar respaldada por supuestos explícitos. Los siguientes son los supuestos de este proyecto, identificados con códigos que se referencian a lo largo del notebook.

| ID | Supuesto |
|---|---|
| **S1** | La clave primaria del modelo es `(num_doc, obl17, f_analisis)` — una fila por cliente, obligación y periodo mensual |
| **S2** | `num_doc` y `obl17` son identificadores anonimizados (hashes). No se infiere ningún atributo del cliente desde ellos |
| **S3** | `var_rta` se trata como estable por obligación; si varía entre periodos se usa el valor observado en ese corte |
| **S4** | Las obligaciones ausentes en `canales` (79.3%) representan **cero actividad transaccional** en ese periodo → imputar con 0 |
| **S5** | Los nulos en `excedentes` crecen con la ventana temporal (3m→12m) por obligaciones jóvenes sin historial → imputar con 0 |
| **S6** | `moras` no tiene ventana de 12m — se trata como feature faltante (posiblemente no disponible en producción) |
| **S7** | Split temporal: Train 8 periodos / Test 5 periodos / OOT 4 periodos. Justificación: preservar orden cronológico sin leakage |
| **S8** | El desbalance de clases (≈79%/21%) requiere estratificación al entrenar y métricas robustas (AUC, KS, F1) |
| **S9** | Las columnas `pago_debito_*` en `pagos` son señal directa de `var_rta` — riesgo de **data leakage** si se usan sin cuidado |

## 3. Análisis Individual de las Fuentes CSV

In [ ]:
# Carga de las 6 fuentes (S1, S2)
sources = load_raw_sources(DATALAKE)

# Resumen de carga
print('=' * 65)
print(f'{'FUENTE':<14} {'FILAS':>8} {'COLS':>7} {'NULOS_TOT':>10} {'PERIODOS':>9}')
print('=' * 65)
for alias, df in sources.items():
    nulos = df.isnull().sum().sum()
    periodos = df['f_analisis'].nunique()
    print(f'{alias:<14} {len(df):>8,} {len(df.columns):>7} {nulos:>10,} {periodos:>9}')

### 3.1 Tabla Maestra — `clientes`

Es la tabla ancla del modelo: contiene la variable respuesta `var_rta` y define el universo de obligaciones a analizar.

In [ ]:
cli = sources['clientes']

print('Muestra:')
display(cli.head(4))

print(f'\nFilas   : {len(cli):,}')
print(f'Cols    : {list(cli.columns)}')
print(f'PK únicas (num_doc, obl17): {cli[["num_doc", "obl17"]].drop_duplicates().shape[0]:,}')
print(f'Periodos: {sorted(cli["f_analisis"].dt.strftime("%Y-%m").unique().tolist())}')

In [ ]:
# Distribución de la variable respuesta
dist = cli[TARGET_COL].value_counts()
pct  = cli[TARGET_COL].value_counts(normalize=True) * 100

print('Distribución de var_rta:')
print(f'  Clase 1 (débito exclusivo + recurrencia ≥40%) : {dist[1]:>7,} ({pct[1]:.1f}%)')
print(f'  Clase 0 (otros canales o combinación)         : {dist[0]:>7,} ({pct[0]:.1f}%)')
print(f'  Ratio desbalance                              : {pct[1]/pct[0]:.1f}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barras
axes[0].bar(['Clase 0\n(otros canales)', 'Clase 1\n(débito exclusivo)'],
            [dist[0], dist[1]], color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribución de var_rta', fontweight='bold')
axes[0].set_ylabel('Número de registros')
for i, v in enumerate([dist[0], dist[1]]):
    axes[0].text(i, v + 200, f'{v:,}\n({[pct[0], pct[1]][i]:.1f}%)', ha='center', fontsize=10)

# Evolución por periodo
evo = cli.groupby(['f_analisis', TARGET_COL]).size().unstack(fill_value=0)
evo.plot(ax=axes[1], color=['#e74c3c', '#2ecc71'], linewidth=2, marker='o', markersize=5)
axes[1].set_title('Evolución del target por periodo', fontweight='bold')
axes[1].set_ylabel('Registros')
axes[1].set_xlabel('')
axes[1].legend(['Clase 0', 'Clase 1'])
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIGURES / 'debitos_dist_target.png', dpi=150, bbox_inches='tight')
plt.show()
print('S8: desbalance ≈79%/21% → requiere estratificación al entrenar.')

### 3.2 Pagos por Canal — `pagos` (tanque)

Agrupación de pagos por tipo de canal (débito, físico, virtual, otros). Contiene la **señal más directa del target** y estadísticas rolling en ventanas de 3, 6, 9 y 12 meses.

In [ ]:
pag = sources['pagos']

# Tipos de pago disponibles
tipos = ['debito', 'fisico', 'virtual', 'otros']
ventanas = ['3m', '6m', '9m', '12m']

print(f'Filas  : {len(pag):,} | Cols: {len(pag.columns)}')
print(f'Patrón : avg/min/max/stddev_pago_[{"|" .join(tipos)}]_[{"|" .join(ventanas)}]')
print('\nEstadísticas avg_pago_debito_3m (señal directa de var_rta — S9):')
display(pag['avg_pago_debito_3m'].describe().to_frame().T)

# Porcentaje de registros con débito > 0
con_debito = (pag['avg_pago_debito_3m'] > 0).mean() * 100
print(f'\nRegistros con avg_pago_debito_3m > 0: {con_debito:.1f}%')
print('\n⚠ ALERTA S9: pago_debito_* es señal directa de var_rta. Evaluar exclusión antes de entrenar.')

In [ ]:
# Comparación débito vs otros canales por tipo de target (en el merge con clientes)
tmp = cli.merge(pag[JOIN_KEYS + ['avg_pago_debito_3m', 'avg_pago_fisico_3m',
                                  'avg_pago_virtual_3m', 'avg_pago_otros_3m']],
                on=JOIN_KEYS, how='left')

medias = tmp.groupby(TARGET_COL)[['avg_pago_debito_3m', 'avg_pago_fisico_3m',
                                    'avg_pago_virtual_3m', 'avg_pago_otros_3m']].mean()

fig, ax = plt.subplots(figsize=(10, 4))
labels = ['Débito 3m', 'Físico 3m', 'Virtual 3m', 'Otros 3m']
x = np.arange(len(labels))
w = 0.35
ax.bar(x - w/2, medias.loc[0], w, label='Clase 0 (otros canales)', color='#e74c3c', alpha=0.85)
ax.bar(x + w/2, medias.loc[1], w, label='Clase 1 (débito exclusivo)', color='#2ecc71', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_title('Media de pagos por tipo de canal (3m) según target', fontweight='bold')
ax.set_ylabel('Monto promedio')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig(FIGURES / 'debitos_pagos_por_target.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Transacciones por Canal — `canales`

Fuente más amplia en dimensionalidad: **4,258 columnas** que combinan canal, tipo de operación, estadística agregada (avg/min/max/stddev/sum) y ventana temporal (ult3, ult6, etc.). Cubre solo el **20.7%** del universo de obligaciones.

In [ ]:
can = sources['canales']

non_key = [c for c in can.columns if c not in JOIN_KEYS]
prefijos = pd.Series([c.split('_')[0] for c in non_key]).value_counts()

print(f'Filas: {len(can):,} | Cols: {len(can.columns):,}')
print(f'Obligaciones únicas (num_doc, obl17): {can[["num_doc", "obl17"]].drop_duplicates().shape[0]:,}')
print(f'Cobertura vs universo: {can[["num_doc", "obl17"]].drop_duplicates().shape[0] / cli[["num_doc","obl17"]].drop_duplicates().shape[0]*100:.1f}%')
print(f'Nulos en features: {can[non_key].isnull().sum().sum():,} '
      f'({can[non_key].isnull().mean().mean()*100:.1f}% de celdas)')

print('\nDistribución de prefijos de columnas:')
for pref, cnt in prefijos.items():
    print(f'  {pref:<8}: {cnt:>4} columnas')

print('\nCanales identificados:')
canales_id = ['app_per', 'app_pyme', 'bill_mvl', 'btn_bco', 'cajero',
              'cr_bcrio', 'pos', 'pse_emp', 'pse_per', 'rec_e_v',
              'suc_fis', 'suc_tel', 'sv_pyme', 'sve', 'svp']
print('  ' + ', '.join(canales_id))

print('\nS4: obligaciones sin fila en canales = sin actividad transaccional → se imputará con 0.')

### 3.4 Excedentes de Pago — `excedentes`

Captura cuánto pagan los clientes **por encima de la cuota** y el porcentaje que representa el pago respecto al valor de cuota. Presente para todas las obligaciones con nulos crecientes en ventanas largas (S5).

In [ ]:
exc = sources['excedentes']

# Nulos por ventana
ventanas_exc = ['3m', '6m', '9m', '12m']
null_porc = {v: exc[[c for c in exc.columns if f'porc_pago_{v}' in c]].isnull().mean().mean() * 100
             for v in ventanas_exc}

print(f'Filas: {len(exc):,} | Cols: {len(exc.columns)}')
print('\nNulos en columnas porc_pago por ventana temporal (S5):')
for v, pct_n in null_porc.items():
    barra = '█' * int(pct_n / 0.5)
    print(f'  {v:>4} : {pct_n:5.2f}%  {barra}')

print('\nObservación: nulos crecen con la ventana → obligaciones sin historial completo.')
print('S5: se imputará con 0 (sin excedente registrado = excedente cero).')

# Estadísticas de avg_excedente_pago_3m
print('\nEstadísticas avg_excedente_pago_3m:')
display(exc['avg_excedente_pago_3m'].describe().to_frame().T)

### 3.5 Gestiones de Cobranza — `gestiones`

Historial de gestiones realizadas sobre cada obligación: cantidad de gestiones, respuestas por cliente (RPC), acuerdos de pago, promesas cumplidas e intensidad de gestión (rank). Sin valores nulos.

In [ ]:
gest = sources['gestiones']

grupos_gest = {
    'cant_gestiones'    : [c for c in gest.columns if 'cant_gestiones' in c],
    'cant_rpc'          : [c for c in gest.columns if 'cant_rpc' in c],
    'cant_acuerdo'      : [c for c in gest.columns if 'cant_acuerdo' in c],
    'promesas_cumplidas': [c for c in gest.columns if 'promesas_cumplidas' in c],
    'maximo_rank'       : [c for c in gest.columns if 'maximo_rank' in c],
}

print(f'Filas: {len(gest):,} | Cols: {len(gest.columns)} | Nulos: {gest.isnull().sum().sum()}')
print('\nGrupos de features en gestiones:')
for grp, cols in grupos_gest.items():
    print(f'  {grp:<22}: {len(cols)} cols (avg/min/max/stddev × 3m/6m/9m/12m)')

# Comparar gestiones entre clases
tmp_g = cli[['num_doc', 'obl17', 'f_analisis', TARGET_COL]].merge(
    gest[JOIN_KEYS + ['avg_cant_gestiones_3m', 'avg_cant_rpc_3m', 'avg_promesas_cumplidas_3m']],
    on=JOIN_KEYS, how='left'
)

print('\nMedia de indicadores de gestión por clase de var_rta:')
display(
    tmp_g.groupby(TARGET_COL)[['avg_cant_gestiones_3m', 'avg_cant_rpc_3m', 'avg_promesas_cumplidas_3m']].mean()
)

### 3.6 Mora por Obligación — `moras`

Estadísticas de días en mora para cada obligación en ventanas de 3, 6 y 9 meses. **No tiene ventana de 12m** (S6).

In [ ]:
mor = sources['moras']

print(f'Filas: {len(mor):,} | Cols: {len(mor.columns)} | Nulos: {mor.isnull().sum().sum()}')
print(f'Columnas: {[c for c in mor.columns if c not in JOIN_KEYS]}')
print('S6: ventana 12m ausente — posiblemente no disponible en producción.')

# Distribución de mora promedio 3m
tmp_m = cli[['num_doc', 'obl17', 'f_analisis', TARGET_COL]].merge(
    mor[JOIN_KEYS + ['moras_avg_mora_3m', 'moras_max_mora_3m']], on=JOIN_KEYS, how='left'
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, clase in enumerate([0, 1]):
    datos = tmp_m[tmp_m[TARGET_COL] == clase]['moras_avg_mora_3m'].dropna()
    axes[i].hist(datos, bins=30, color=['#e74c3c', '#2ecc71'][i], alpha=0.8, edgecolor='white')
    axes[i].set_title(f'Mora promedio 3m — Clase {clase}', fontweight='bold')
    axes[i].set_xlabel('Días de mora (promedio 3m)')
    axes[i].set_ylabel('Frecuencia')
    axes[i].axvline(datos.median(), color='black', ls='--', label=f'Mediana: {datos.median():.1f}')
    axes[i].legend()

plt.suptitle('Distribución de mora por clase de target', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / 'debitos_mora_por_target.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 4. Proceso de Unificación — Paso a Paso

La unificación sigue el principio de **menor pérdida de información** usando `LEFT JOIN` desde la tabla base (`clientes`) hacia las fuentes de features. La llave compuesta garantiza la granularidad correcta (S1).

In [ ]:
print('Llave primaria (S1):', JOIN_KEYS)
print('Estrategia de join : LEFT JOIN desde clientes hacia cada fuente')
print('Tabla base         : clientes (tiene var_rta)')
print()

# Verificar unicidad de la llave en cada fuente
print('Verificación de unicidad de PK por fuente:')
print(f'  {'Fuente':<14} {'Total filas':>12} {'PK únicas':>10} {'Duplicadas':>11}')
print('  ' + '-' * 50)
for alias, df in sources.items():
    total = len(df)
    unicas = df[JOIN_KEYS].drop_duplicates().shape[0]
    dup = total - unicas
    flag = ' ⚠' if dup > 0 else ' ✓'
    print(f'  {alias:<14} {total:>12,} {unicas:>10,} {dup:>11,}{flag}')

### 4.1 Tratamiento de Duplicados en las Fuentes CSV

Antes de ejecutar los JOINs se detectaron **filas con clave compuesta repetida** en 5 de las 6 fuentes. El diagnóstico distingue dos categorías:

| Fuente | Filas brutas | Claves dup. | Filas dup. | Idénticas (all cols) | Distintas (same key, diff values) |
|---|---|---|---|---|---|
| clientes | 46,736 | 0 | 0 | 0 | 0 |
| excedentes | 47,194 | 458 | 791 | 791 | 0 |
| gestiones | 47,194 | 458 | 791 | 791 | 0 |
| moras | 47,194 | 458 | 791 | 791 | 0 |
| pagos | 47,194 | 458 | 791 | 791 | 0 |
| canales | 10,801 | 1,294 | 1,382 | 360 | 1,022 |

#### Caso simple — excedentes, gestiones, moras, pagos

Todas las filas duplicadas son **copias exactas en la totalidad de columnas** (mismo valor en cada campo). La estrategia es inequívoca: `df.drop_duplicates()` sin restricción de columnas.  
Impacto: **−458 filas** por archivo antes de insertar en base de datos.

#### Caso complejo — canales

Se detectaron dos sub-casos:

- **360 filas**: copias exactas → tratamiento idéntico al caso simple (`drop_duplicates()`).
- **1,022 filas** pertenecientes a **24 grupos** con la misma clave compuesta pero **valores distintos en ~1,181 columnas**.

Ejemplo ilustrativo de un grupo conflictivo:

| Campo | Versión A | Versión B |
|---|---|---|
| `num_doc` | `0f7741b0b059ab7` | `0f7741b0b059ab7` |
| `obl17` | `0a1ca04ecbd413125` | `0a1ca04ecbd413125` |
| `f_analisis` | `2025-07-01` | `2025-07-01` |
| `trx_mnt_total` | 19,765,290 | 4,293,344 |
| `trx_cnt_total` | 16 | 12 |
| `trx_cnt_app_per_pag_ob` | 1 | 0 |
| `trx_cnt_app_per_trf` | 0 | 9 |

El patrón observado: las dos versiones **se repiten alternadamente** y no son subconjuntos complementarios evidentes.

#### Opciones evaluadas para los 24 grupos conflictivos

| Opción | Estrategia | Supuesto implícito | Riesgo |
|---|---|---|---|
| **a) Sumar columnas numéricas** | Agregar A + B por columna | A y B son cortes complementarios del mismo periodo | Puede inflar artificialmente los totales si no son complementarios |
| **b) Conservar mayor trx_mnt_total** | Quedarse con la versión más grande | Una versión es más completa que la otra | Sesgo de selección; descarta transacciones reales |
| **c) Excluir los 24 grupos** | Eliminar todas las filas de esas 24 claves | El origen es ambiguo y no resoluble sin la fuente | Pérdida mínima: 24 / 9,460 obligaciones únicas = **0.25%** |
| **d) Conservar primera / última** | `groupby(...).first()` o `.last()` | El archivo fue reprocesado y la última fila supersede | Introduce ruido si no hay garantía de orden de carga |

#### Decisión adoptada — **Opción c: exclusión de los 24 grupos conflictivos**

**Justificación:**
- Las versiones A y B no son complementarias (totales divergen, patrones de canal distintos).
- Sin información sobre el proceso ETL de origen no es posible determinar cuál versión es correcta.
- El impacto es mínimo: 24 grupos = 0.25% del universo de obligaciones con actividad en canales.
- La opción c es la más conservadora desde el punto de vista de calidad de datos: **es preferible no tener el dato que tener el dato incorrecto** en un modelo de crédito.

> **Supuesto añadido S11:** Los 24 grupos de `canales` con valores conflictivos en la clave compuesta son excluidos del modelo analítico. Para esas 24 obligaciones, las columnas de canales se tratan como sin actividad transaccional (igual que S4).

In [ ]:
# Diagnóstico reproductible de duplicados en las 6 fuentes CSV
KEY_COLS = ['num_doc', 'obl17', 'f_analisis']

print(f'{"Fuente":<14} {"Filas":>8} {"Claves dup":>11} {"Filas dup":>10} '
      f'{"Idénticas":>10} {"Distintas":>10}')
print('-' * 66)

for alias, df_src in sources.items():
    total = len(df_src)

    # Filas exactamente duplicadas (todas las columnas)
    exactas = total - len(df_src.drop_duplicates())
    df_dedup = df_src.drop_duplicates()

    # Duplicados residuales en la clave después de dedup exacto
    dup_mask = df_dedup.duplicated(subset=KEY_COLS, keep=False)
    filas_dup_clave = dup_mask.sum()
    claves_dup = df_dedup[dup_mask][KEY_COLS].drop_duplicates().shape[0] if filas_dup_clave else 0
    distintas = filas_dup_clave  # las que quedan después de exactas = conflictivas

    flag = ' ⚠' if (exactas + distintas) > 0 else ' ✓'
    print(f'{alias:<14} {total:>8,} {claves_dup:>11,} {exactas + distintas:>10,} '
          f'{exactas:>10,} {distintas:>10,}{flag}')

print()
print('Estrategia aplicada en loader.py:')
print('  · Caso simple (filas idénticas)   → df.drop_duplicates()')
print('  · Caso complejo (canales, 24 grp) → opción c: excluir grupos conflictivos')
print('  · Impacto final en canales        : 24 / 9,460 obligaciones únicas = 0.25%')
print()
print('S11: los 24 grupos excluidos de canales se tratan como sin actividad (S4).')

### 4.2 Cobertura de obligaciones entre fuentes

In [ ]:
universo = set(zip(cli['num_doc'], cli['obl17']))

print(f'Universo de obligaciones (num_doc, obl17) en clientes: {len(universo):,}')
print()

cobertura = {}
for alias, df in sources.items():
    if alias == 'clientes':
        continue
    fuente_keys = set(zip(df['num_doc'], df['obl17']))
    intersec = len(universo & fuente_keys)
    solo_fuente = len(fuente_keys - universo)
    pct = intersec / len(universo) * 100
    cobertura[alias] = pct
    print(f'{alias:<14} cobertura={pct:.1f}% | en fuente no en clientes={solo_fuente}')

# Visualización de cobertura
fig, ax = plt.subplots(figsize=(8, 4))
fuentes = list(cobertura.keys())
pcts = list(cobertura.values())
colores = ['#2ecc71' if p == 100 else '#e67e22' for p in pcts]
bars = ax.barh(fuentes, pcts, color=colores, edgecolor='white')
ax.set_xlim(0, 110)
ax.set_xlabel('Cobertura (%)')
ax.set_title('Cobertura de obligaciones por fuente\n(respecto al universo en clientes)', fontweight='bold')
for bar, pct in zip(bars, pcts):
    ax.text(pct + 1, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontweight='bold')
ax.axvline(100, color='gray', ls='--', alpha=0.5)
plt.tight_layout()
plt.savefig(FIGURES / 'debitos_cobertura_fuentes.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nS4: canales cubre solo ~20.7% → JOIN izquierdo + imputación con 0 para el 79.3% restante.')

### 4.3 Ejecución del modelo analítico unificado

El proceso encapsula: carga → deduplicación → LEFT JOINs → imputación → validación.

In [ ]:
print('Ejecutando build_analytical_model()...')
print('Pasos internos:')
print('  1. Carga de 6 CSV (load_raw_sources)')
print('  2. LEFT JOIN: clientes ← excedentes (cobertura 100%)')
print('  3. LEFT JOIN: base ← gestiones    (cobertura 100%)')
print('  4. LEFT JOIN: base ← moras        (cobertura 100%)')
print('  5. LEFT JOIN: base ← pagos        (cobertura 100%)')
print('  6. LEFT JOIN: base ← canales      (cobertura ~20.7%) → fill(0) en cols trx_* (S4)')
print('  7. Imputación excedentes nulos con 0 (S5)')
print()

df = build_analytical_model(DATALAKE)

print()
print(f'Dataset consolidado: {len(df):,} filas × {len(df.columns):,} columnas')

### 4.4 Validación de integridad del join

In [ ]:
report = validate_join_integrity(df)

print('=' * 55)
print('REPORTE DE VALIDACIÓN DE INTEGRIDAD')
print('=' * 55)
print(f'Total filas          : {report["total_filas"]:,}')
print(f'Claves PK únicas     : {report["claves_unicas"]:,}')
print(f'Claves duplicadas    : {report["claves_duplicadas"]} {"✓" if report["claves_duplicadas"] == 0 else "⚠"}')
print(f'Nulos residuales     : {len(report["nulos_restantes"])} cols {"✓" if not report["nulos_restantes"] else "⚠"}')
print(f'Cobertura canales    : {report["cobertura_canales_pct"]}%')
print(f'Dist. target         : {report["distribucion_target"]}')
print(f'Periodos presentes   : {len(report["periodos"])} ({report["periodos"][0]} … {report["periodos"][-1]})')
print(f'\nValidación SUPERADA  : {report["ok"]} {"✅" if report["ok"] else "❌"}')

---

## 5. Caracterización del Dataset Consolidado

In [ ]:
# Trazabilidad por grupo de features
groups = get_feature_columns(df)

total_features = sum(len(v) for v in groups.values())

print('Inventario de features por fuente:')
print(f'  {'Fuente':<14} {'Features':>9} {'%':>7}')
print('  ' + '-' * 33)
for fuente, cols in groups.items():
    pct_f = len(cols) / total_features * 100
    barra = '▪' * int(pct_f / 2)
    print(f'  {fuente:<14} {len(cols):>9,}  {pct_f:5.1f}%  {barra}')
print('  ' + '-' * 33)
print(f'  {"TOTAL":<14} {total_features:>9,}  100.0%')
print(f'\n  + 4 columnas de metadata ({JOIN_KEYS + [TARGET_COL]})')
print(f'  = {len(df.columns):,} columnas totales en el dataset')

In [ ]:
# Gráfico de distribución de features por fuente
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Torta
fuentes_names = list(groups.keys())
sizes = [len(groups[f]) for f in fuentes_names]
colores_pie = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
wedges, texts, autotexts = axes[0].pie(
    sizes, labels=fuentes_names, autopct='%1.1f%%',
    colors=colores_pie, startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(9)
axes[0].set_title('Distribución de features por fuente', fontweight='bold')

# Barras horizontales con conteo
y_pos = range(len(fuentes_names))
axes[1].barh(list(y_pos), sizes, color=colores_pie, edgecolor='white')
axes[1].set_yticks(list(y_pos))
axes[1].set_yticklabels(fuentes_names)
axes[1].set_xlabel('Número de features')
axes[1].set_title('Conteo de features por fuente', fontweight='bold')
for i, v in enumerate(sizes):
    axes[1].text(v + 10, i, f'{v:,}', va='center')

plt.tight_layout()
plt.savefig(FIGURES / 'debitos_features_por_fuente.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Evolución temporal del dataset
evolucion = df.groupby('f_analisis').agg(
    registros=(TARGET_COL, 'count'),
    clase_1=(TARGET_COL, 'sum'),
    clase_0=(TARGET_COL, lambda x: (x == 0).sum()),
).reset_index()
evolucion['pct_clase_1'] = evolucion['clase_1'] / evolucion['registros'] * 100

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# Registros por periodo
axes[0].bar(evolucion['f_analisis'], evolucion['clase_1'],
            label='Clase 1 (débito exclusivo)', color='#2ecc71', alpha=0.85)
axes[0].bar(evolucion['f_analisis'], evolucion['clase_0'],
            bottom=evolucion['clase_1'], label='Clase 0 (otros)', color='#e74c3c', alpha=0.85)
axes[0].set_ylabel('Registros')
axes[0].set_title('Volumen y composición del dataset por periodo', fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# % clase 1 por periodo
axes[1].plot(evolucion['f_analisis'], evolucion['pct_clase_1'],
             color='#2ecc71', linewidth=2.5, marker='o', markersize=6)
axes[1].axhline(evolucion['pct_clase_1'].mean(), color='gray', ls='--',
                label=f'Media: {evolucion["pct_clase_1"].mean():.1f}%')
axes[1].set_ylim(0, 100)
axes[1].set_ylabel('% Clase 1')
axes[1].set_xlabel('Periodo (f_analisis)')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIGURES / 'debitos_evolucion_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Estadísticas descriptivas del dataset consolidado (muestra de features clave)
features_clave = [
    'avg_pago_debito_3m', 'avg_pago_fisico_3m', 'avg_pago_virtual_3m',
    'avg_excedente_pago_3m', 'avg_cant_gestiones_3m',
    'moras_avg_mora_3m', 'trx_mnt_total',
]

features_presentes = [f for f in features_clave if f in df.columns]

print('Estadísticas descriptivas — features clave del dataset consolidado:')
display(
    df.groupby(TARGET_COL)[features_presentes].describe().T
    .loc[(slice(None), ['mean', 'std', '50%']), :]
    .rename(index={'50%': 'median'})
)

---

## 6. Split Temporal Train / Test / OOT

El esquema de evaluación temporal garantiza que el modelo **nunca ve datos del futuro durante el entrenamiento**, lo que es crítico en contextos de riesgo bancario (S7).

| Partición | Fechas | Periodos | Rol |
|---|---|---|---|
| **Train** | 2024-07-01 → 2025-02-01 | 8 | Ajuste del modelo |
| **Test** | 2025-03-01 → 2025-07-01 | 5 | Selección de modelo e hiperparámetros |
| **OOT** | 2025-08-01 → 2025-11-01 | 4 | Evaluación fuera de muestra (evaluador externo) |

In [ ]:
df_train, df_test, df_oot = split_train_test_oot(df)

splits = {'Train': df_train, 'Test': df_test, 'OOT': df_oot}

print('=' * 65)
print(f'{'Split':<8} {'Filas':>7} {'Periodos':>9} {'Desde':>12} {'Hasta':>12} {'%Clase1':>8}')
print('=' * 65)
for nombre, part in splits.items():
    periodos = sorted(part['f_analisis'].dt.strftime('%Y-%m-%d').unique())
    pct1 = part[TARGET_COL].mean() * 100
    print(f'{nombre:<8} {len(part):>7,} {len(periodos):>9} {periodos[0]:>12} {periodos[-1]:>12} {pct1:>7.1f}%')

In [ ]:
# Visualización del split temporal sobre la línea de tiempo
fig, ax = plt.subplots(figsize=(13, 3))

colores_split = {'Train': '#3498db', 'Test': '#f39c12', 'OOT': '#e74c3c'}
y = 0.5

for nombre, part in splits.items():
    periodos = sorted(part['f_analisis'].unique())
    inicio = periodos[0]
    fin    = periodos[-1]
    ax.barh(y, (fin - inicio).days, left=inicio,
            height=0.4, color=colores_split[nombre], alpha=0.85,
            label=f'{nombre} ({len(part):,} filas)', edgecolor='white')
    ax.text(inicio + (fin - inicio) / 2, y,
            f'{nombre}\n{len(periodos)} periodos', ha='center', va='center',
            fontweight='bold', fontsize=9, color='white')

ax.set_yticks([])
ax.set_xlabel('Periodo (f_analisis)')
ax.set_title('Esquema temporal Train / Test / OOT (S7)', fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1, 1.35))
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(plt.matplotlib.dates.MonthLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES / 'debitos_split_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

print('S7: particiones definidas exclusivamente por f_analisis → sin solapamiento temporal.')

In [ ]:
# Balance del target por partición
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (nombre, part) in zip(axes, splits.items()):
    dist_p = part[TARGET_COL].value_counts()
    ax.pie([dist_p.get(0, 0), dist_p.get(1, 0)],
           labels=['Clase 0', 'Clase 1'],
           autopct='%1.1f%%',
           colors=['#e74c3c', '#2ecc71'],
           wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    ax.set_title(f'{nombre}\n({len(part):,} filas)', fontweight='bold')

plt.suptitle('Distribución del target por partición (S8)', fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(FIGURES / 'debitos_balance_por_split.png', dpi=150, bbox_inches='tight')
plt.show()

print('S8: la distribución natural del target se preserva en cada partición.')
print('     Al entrenar se aplicará estratificación (class_weight o SMOTE).')

---

## 7. Conclusiones y Alertas

### 7.1 Resumen del Dataset Consolidado

In [ ]:
print('=' * 60)
print('RESUMEN DEL DATASET ANALÍTICO CONSOLIDADO')
print('=' * 60)
print(f'Granularidad    : cliente × obligación × periodo (mensual)')
print(f'Llave primaria  : {JOIN_KEYS}')
print(f'Total filas     : {len(df):,}')
print(f'Total columnas  : {len(df.columns):,}')
print(f'  - Metadata    : 4  (num_doc, obl17, f_analisis, var_rta)')
print(f'  - Features    : {len(df.columns) - 4:,}')
print(f'Periodos        : {df["f_analisis"].nunique()} (Jul 2024 – Nov 2025)')
print(f'Obligaciones    : {df[["num_doc","obl17"]].drop_duplicates().shape[0]:,} únicas')
print(f'Target (clase1) : {df[TARGET_COL].mean()*100:.1f}% | desbalance 3.7:1')
print()
print('Features por fuente:')
for fuente, cols in get_feature_columns(df).items():
    print(f'  {fuente:<14}: {len(cols):,}')

### 7.2 Supuestos Validados

| ID | Supuesto | Estado |
|---|---|---|
| S1 | PK = (num_doc, obl17, f_analisis) | ✅ Validado — 0 duplicados post-join |
| S2 | Identificadores son hashes anónimos | ✅ Confirmado — longitud uniforme, sin patrón inferible |
| S3 | var_rta estable por obligación en el periodo | ✅ Asumido — no hay información que lo contradiga |
| S4 | Canales ausentes = sin actividad → fill(0) | ✅ Aplicado — 19.6% de filas con actividad en canales |
| S5 | Nulos en excedentes por historial insuficiente → fill(0) | ✅ Aplicado — 0 nulos residuales post-imputación |
| S6 | Moras sin ventana 12m — feature faltante | ⚠️ Pendiente confirmar si disponible en producción |
| S7 | Split temporal Train/Test/OOT por f_analisis | ✅ Implementado — particiones sin solapamiento |
| S8 | Desbalance requiere tratamiento al entrenar | ⚠️ Pendiente — aplicar class_weight o SMOTE en pipeline de entrenamiento |
| S9 | pago_debito_* es señal directa del target | ⚠️ **ALERTA ACTIVA** — evaluar exclusión o diseño de features lagged |

### 7.3 Alertas Técnicas

1. **Data Leakage (S9):** Las columnas `avg/min/max_pago_debito_*` en `pagos` son construcciones estadísticas derivadas directamente del comportamiento de débito que define `var_rta`. Si estas variables se calculan sobre el mismo periodo de etiquetado, el modelo aprenderá la tautología, no el patrón. **Acción requerida:** verificar que las ventanas temporales de estas variables son previas al periodo de corte (`f_analisis`).

2. **Dimensionalidad extrema en canales (S4):** 4,255 features de una fuente con solo 20.7% de cobertura introduce riesgo de overfitting y alta esparsidad. **Acción requerida:** selección de features (varianza cero, correlación con target, importancia por árbol).

3. **Desbalance de clases (S8):** Ratio 3.7:1 (clase 1 dominante). Un modelo naive que predice siempre clase 1 tendría accuracy del 78.8% sin aprender nada. **Acción requerida:** usar AUC-ROC + KS como métricas primarias, no accuracy.

### 7.4 Próximos Pasos

1. **Feature Engineering:** construir variables derivadas que capturen exclusividad de canal y recurrencia de débito desde `pagos` y `canales`
2. **Selección de features:** reducir las 4,255 columnas de canales usando importancia y correlación
3. **Modelado:** entrenar clasificadores binarios (Logistic Regression, Random Forest, XGBoost) con validación temporal y estratificación
4. **Evaluación:** comparar AUC, KS, Precision@Recall usando las particiones Test y OOT
5. **Tablero:** exportar resultados a Metabase para visualización de indicadores de negocio

In [ ]:
import os

print('Figuras generadas en este notebook:')
for f in sorted(FIGURES.glob('debitos_*.png')):
    print(f'  {f}')